### 1 - Configuration

In [0]:
# ============================================================
# IESO Demand → Bronze Auto Loader ingestion
# Source: CSV files in /Volumes/ogp_dev/landing/zone/ieso/demand/
# Target: ogp_dev.bronze.ieso_demand_raw (Delta table)
# ============================================================

import uuid

# Target table (three-part name)
CATALOG = "ogp_dev"
SCHEMA  = "bronze"
TABLE   = "ieso_demand_raw"
TABLE_FQN = f"{CATALOG}.{SCHEMA}.{TABLE}"

# Source files
LANDING_PATH = "/Volumes/ogp_dev/landing/zone/ieso/demand"

# Auto Loader state — checkpoint and schema
CHECKPOINT_PATH = f"/Volumes/ogp_dev/ops/checkpoints/bronze/{TABLE}"
SCHEMA_PATH     = f"{CHECKPOINT_PATH}/_schema"

# Run identifier — in production this would be a Lakeflow Job run ID
RUN_ID = f"manual_{uuid.uuid4()}"

print(f"Source path:      {LANDING_PATH}")
print(f"Target table:     {TABLE_FQN}")
print(f"Checkpoint path:  {CHECKPOINT_PATH}")
print(f"Schema path:      {SCHEMA_PATH}")
print(f"Run ID:           {RUN_ID}")

Source path:      /Volumes/ogp_dev/landing/zone/ieso/demand
Target table:     ogp_dev.bronze.ieso_demand_raw
Checkpoint path:  /Volumes/ogp_dev/ops/checkpoints/bronze/ieso_demand_raw
Schema path:      /Volumes/ogp_dev/ops/checkpoints/bronze/ieso_demand_raw/_schema
Run ID:           manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1


### 2 - Auto Loader read stream

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

# Schema hints lock in types for known columns.
# Backticks are required for column names with spaces.
SCHEMA_HINTS = """
  Date DATE,
  Hour INT,
  `Market Demand` INT,
  `Ontario Demand` INT
"""

raw_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",          "csv")
        .option("cloudFiles.schemaLocation",  SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes","true")
        .option("cloudFiles.schemaHints",     SCHEMA_HINTS)
        .option("header",                     "true")
        .option("comment",                    "\\")     # IESO preamble lines
        .option("encoding",                   "UTF-8")
        .load(LANDING_PATH)
)

# Preview inferred + hinted schema
print("Auto Loader schema:")
raw_stream.printSchema()

Auto Loader schema:
root
 |-- Date: date (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Market Demand: integer (nullable = true)
 |-- Ontario Demand: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)



### 3 - Transform and enrich

In [0]:
# Rename source columns to SQL-friendly snake_case
# Add audit columns: _ingest_ts, _source_file, _ingest_run_id
bronze_df = (
    raw_stream
        .withColumnRenamed("Date",            "event_date")
        .withColumnRenamed("Hour",            "hour_ending")
        .withColumnRenamed("Market Demand",   "market_demand_mw")
        .withColumnRenamed("Ontario Demand",  "ontario_demand_mw")
        .withColumn("_ingest_ts",     current_timestamp())
        .withColumn("_source_file",   col("_metadata.file_path"))
        .withColumn("_ingest_run_id", lit(RUN_ID))
)

print("Bronze schema (after enrichment):")
bronze_df.printSchema()

Bronze schema (after enrichment):
root
 |-- event_date: date (nullable = true)
 |-- hour_ending: integer (nullable = true)
 |-- market_demand_mw: integer (nullable = true)
 |-- ontario_demand_mw: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _ingest_run_id: string (nullable = false)



### 4 - Write to Delta table

In [0]:
query = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TABLE_FQN)
)

# Wait for the batch-style stream to complete
query.awaitTermination()

print(f"Ingestion complete — table: {TABLE_FQN}")

Ingestion complete — table: ogp_dev.bronze.ieso_demand_raw


### 5 - Verify

In [0]:
# Row count
count = spark.table(TABLE_FQN).count()
print(f"Rows in {TABLE_FQN}: {count}")

# Sample rows
print("\nFirst 5 rows:")
display(spark.table(TABLE_FQN).orderBy("event_date", "hour_ending").limit(5))

# Last 5 rows (sanity check)
print("\nLast 5 rows:")
display(spark.table(TABLE_FQN).orderBy(col("event_date").desc(), col("hour_ending").desc()).limit(5))

# Audit column inspection
print("\nDistinct source files ingested:")
from pyspark.sql.functions import count

print("Distinct source files ingested:")
display(
    spark.table(TABLE_FQN)
        .groupBy("_source_file", "_ingest_run_id")
        .agg(count("*").alias("rows"))
)

Rows in ogp_dev.bronze.ieso_demand_raw: 8784

First 5 rows:


event_date,hour_ending,market_demand_mw,ontario_demand_mw,_rescued_data,_ingest_ts,_source_file,_ingest_run_id
2024-01-01,1,17091,14482,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-01-01,2,16658,14180,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-01-01,3,16233,13722,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-01-01,4,15909,13637,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-01-01,5,15998,13697,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1



Last 5 rows:


event_date,hour_ending,market_demand_mw,ontario_demand_mw,_rescued_data,_ingest_ts,_source_file,_ingest_run_id
2024-12-31,24,17247,14350,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-12-31,23,17159,14734,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-12-31,22,17453,15272,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-12-31,21,18124,15767,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1
2024-12-31,20,18718,16373,null,2026-05-15T18:30:45.328Z,/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1



Distinct source files ingested:
Distinct source files ingested:


_source_file,_ingest_run_id,rows
/Volumes/ogp_dev/landing/zone/ieso/demand/PUB_Demand_2024.csv,manual_5d0d88d7-6697-420e-ab94-3eb3e0a1ead1,8784
